<a href="https://colab.research.google.com/github/priyanshi-nigam123/Paraphrase-generation-using-t5/blob/main/Paraphrase_generation_using_t5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# !pip install datasets

In [ ]:
!pip uninstall -y torchvision -q

In [ ]:
!rm -rf /usr/local/lib/python3.12/dist-packages/torchvision*

In [27]:
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments, TrainerCallback
import os
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import datasets.config
datasets.config.TORCHVISION_AVAILABLE = False

In [ ]:
dataset = load_dataset("google-research-datasets/paws", "labeled_final")

def preprocess_paws(dataset, label=1):
  df = pd.DataFrame(dataset)
  df = df[df['label']==label]

  df['input_text'] = "paraphrase :" + df['sentence1']
  df['target_text'] = df['sentence2']

  return df[['input_text','target_text']]

train_df = preprocess_paws(dataset['train']).sample(6000, random_state=42)
test_df = preprocess_paws(dataset['test']).sample(500, random_state=42)
validation_df = preprocess_paws(dataset['validation']).sample(500, random_state=42)

README.md:   0%|          | 0.00/9.79k [00:00<?, ?B/s]

labeled_final/train-00000-of-00001.parqu(…): reconstructing file:   0%|          |  0.00B / 8.43MB            

labeled_final/train-00000-of-00001.parqu(…): downloading bytes:           |  0.00B            

labeled_final/test-00000-of-00001.parque(…): reconstructing file:   0%|          |  0.00B / 1.24MB            

labeled_final/test-00000-of-00001.parque(…): downloading bytes:           |  0.00B            

labeled_final/validation-00000-of-00001.(…): reconstructing file:   0%|          |  0.00B / 1.23MB            

labeled_final/validation-00000-of-00001.(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/49401 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/8000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/8000 [00:00<?, ? examples/s]

In [ ]:
train_dataset = Dataset.from_pandas(train_df)
validation_dataset = Dataset.from_pandas(validation_df)
test_dataset = Dataset.from_pandas(test_df)

# Initialize tokenizer and model
model_name = "t5-base"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

# Tokenization function
def tokenize_function(examples):
    max_length = 512  # T5-base typically uses 512 as default

    inputs = tokenizer(examples['input_text'], max_length=max_length, truncation=True, padding="max_length")
    targets = tokenizer(examples['target_text'], max_length=max_length, truncation=True, padding="max_length")
    inputs['labels'] = targets['input_ids']
    return inputs

# Tokenize datasets
train_dataset = train_dataset.map(tokenize_function, batched=True)
validation_dataset = validation_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Set format for PyTorch
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
validation_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

print(f" Train dataset tokenized: {len(train_dataset)} examples")
print(f" Validation dataset tokenized: {len(validation_dataset)} examples")
print(f" Test dataset tokenized: {len(test_dataset)} examples")

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  892MB            

model.safetensors: downloading bytes:           |  0.00B            

...

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Map:   0%|          | 0/6000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

 Train dataset tokenized: 6000 examples
 Validation dataset tokenized: 500 examples
 Test dataset tokenized: 500 examples


In [ ]:
results_dir = "/content/drive/MyDrive/results"
model_dir = "/content/drive/MyDrive/saved_t5_model"

os.makedirs(results_dir, exist_ok=True)
os.makedirs(model_dir, exist_ok=True)

In [10]:
# Define training arguments with smaller batch size
training_args = TrainingArguments(
    output_dir=results_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=4,
    weight_decay=0.01,
    fp16=True,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    save_total_limit=2,
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
)

# Train the model
trainer.train()

# SAVE MODEL
trainer.save_model(model_dir)
tokenizer.save_pretrained(model_dir)
print(f"Model saved to: {model_dir}")

Epoch,Training Loss,Validation Loss
1,0.070642,0.032241
2,0.062684,0.030565
3,0.058984,0.029836
4,0.057081,0.029737


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


Epoch,Training Loss,Validation Loss
1,0.070642,0.032241
2,0.062684,0.030565
3,0.058984,0.029836
4,0.057081,0.029737


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: /content/drive/MyDrive/saved_t5_model


In [12]:
# Install evaluation libraries
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
!pip install evaluate rouge_score sacrebleu -q

import evaluate

bleu = evaluate.load("sacrebleu")
rouge = evaluate.load("rouge")

def evaluate_model(model, tokenizer, test_df, device, max_length=512, sample_size=100):
    model.eval()
    predictions = []
    references = []

    sample_df = test_df.sample(min(sample_size, len(test_df)), random_state=42)

    for _, row in sample_df.iterrows():
        input_text = row['input_text']
        target_text = row['target_text']

        inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=max_length, padding="max_length")
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            output = model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_length=max_length + 20,
                num_beams=5,
                early_stopping=True
            )

        pred = tokenizer.decode(output[0], skip_special_tokens=True)
        predictions.append(pred)
        references.append([target_text])

    bleu_score = bleu.compute(predictions=predictions, references=references)
    rouge_score = rouge.compute(predictions=predictions, references=[r[0] for r in references])

    print(f"BLEU Score: {bleu_score['score']:.2f}")
    print(f"ROUGE-1: {rouge_score['rouge1']:.4f}")
    print(f"ROUGE-2: {rouge_score['rouge2']:.4f}")
    print(f"ROUGE-L: {rouge_score['rougeL']:.4f}")

    return bleu_score, rouge_score

bleu_result, rouge_result = evaluate_model(model, tokenizer, test_df, device)

BLEU Score: 63.90
ROUGE-1: 0.9278
ROUGE-2: 0.7554
ROUGE-L: 0.8450


In [30]:
model.save_pretrained(model_dir)
tokenizer.save_pretrained(model_dir)
print("Saved!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved!


In [31]:
!ls -la /content/drive/MyDrive/saved_t5_model

total 873121
-rw------- 1 root root      1556 Aug  7 17:52 config.json
-rw------- 1 root root       142 Aug  7 17:52 generation_config.json
-rw------- 1 root root 891644712 Aug  7 17:53 model.safetensors
-rw------- 1 root root      2574 Aug  7 17:53 tokenizer_config.json
-rw------- 1 root root   2424334 Aug  7 17:53 tokenizer.json


In [32]:
import torch
from transformers import T5ForConditionalGeneration, T5Tokenizer

model_dir = "/content/drive/MyDrive/saved_t5_model"

model = T5ForConditionalGeneration.from_pretrained(model_dir)
tokenizer = T5Tokenizer.from_pretrained(model_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("Model loaded successfully!")

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Model loaded successfully!


In [34]:
# Free space milne ke baad RAM se directly Drive me save karein
model.save_pretrained("/content/drive/MyDrive/saved_t5_model")
tokenizer.save_pretrained("/content/drive/MyDrive/saved_t5_model")

print("Model successfully saved to Google Drive!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model successfully saved to Google Drive!


In [35]:
import os
from transformers import T5ForConditionalGeneration, T5Tokenizer

# Load the model and tokenizer from the saved directory
model = T5ForConditionalGeneration.from_pretrained(model_dir)
tokenizer = T5Tokenizer.from_pretrained(model_dir)

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

In [37]:
# Define model_dir
model_dir = "/content/drive/MyDrive/saved_t5_model"

In [39]:
import torch
from transformers import T5ForConditionalGeneration, T5Tokenizer

# Load the model and tokenizer from the saved directory
model = T5ForConditionalGeneration.from_pretrained(model_dir)
tokenizer = T5Tokenizer.from_pretrained(model_dir)

# Set the device (GPU if available, otherwise CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Define max_length
max_length = 512  # Same as training

# Preprocessing function for inference
def preprocess_input(sentence):
    return "paraphrase: " + sentence

# Generate paraphrases with corrected num_beams and num_return_sequences
def generate_paraphrase(input_text, model, tokenizer, max_length=512, num_beams=4, num_return_sequences=4, top_k=100, top_p=0.9, temperature=1.0):
    # Preprocess input
    input_text = preprocess_input(input_text)

    # Tokenize input
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=max_length, padding="max_length")

    # Move inputs to the same device as the model
    inputs = {key: value.to(device) for key, value in inputs.items()}

    # Generate paraphrases
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=max_length + 20,
        num_beams=num_beams,
        num_return_sequences=num_return_sequences,
        no_repeat_ngram_size=2,
        num_beam_groups=num_return_sequences,
        diversity_penalty=0.8,
        early_stopping=True,
        trust_remote_code=True
    )

    # Decode generated outputs
    paraphrased_texts = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]
    return paraphrased_texts

# Example sentence
input_sentence = "The quick brown fox jumps over the lazy dog."

# Generate paraphrases
paraphrased_sentences = generate_paraphrase(
    input_sentence, model, tokenizer, max_length=512, num_return_sequences=4
)

# Display results
print(f"Original: {input_sentence}")
for i, paraphrase in enumerate(paraphrased_sentences, 1):
    print(f"Paraphrase {i}: {paraphrase}")

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Original: The quick brown fox jumps over the lazy dog.
Paraphrase 1: The quick brown fox jumps over the lazy dog.
Paraphrase 2: The quick brown fox jumps over the lazy dog.
Paraphrase 3: The quick brown fox jumps over the lazy dog.
Paraphrase 4: The quick brown fox jumps over the lazy dog.


In [40]:
input_sentence = "She enjoys reading books on rainy afternoons."

paraphrased_sentences = generate_paraphrase(
    input_sentence, model, tokenizer, num_return_sequences=4
)

print(f"Original: {input_sentence}")
for i, paraphrase in enumerate(paraphrased_sentences, 1):
    print(f"Paraphrase {i}: {paraphrase}")

Original: She enjoys reading books on rainy afternoons.
Paraphrase 1: She enjoys reading books on rainy afternoons.
Paraphrase 2: On rainy afternoons she enjoys reading books.
Paraphrase 3: She enjoys reading books on rainy afternoons.
Paraphrase 4: She enjoys reading books on rainy afternoons .


In [41]:
input_sentence = "The dog barked loudly at the stranger outside the house."

paraphrased_sentences = generate_paraphrase(
    input_sentence, model, tokenizer, num_return_sequences=4
)

print(f"Original: {input_sentence}")
for i, paraphrase in enumerate(paraphrased_sentences, 1):
    print(f"Paraphrase {i}: {paraphrase}")

Original: The dog barked loudly at the stranger outside the house.
Paraphrase 1: The dog barked loudly at the stranger outside the house.
Paraphrase 2: The dog snarled loudly at the stranger outside the house.
Paraphrase 3: The dog barked loudly at the stranger outside the house.
Paraphrase 4: The dog barked loudly at the stranger outside the house.


In [42]:
input_sentence = "Climate change is one of the most pressing issues of our time."

paraphrased_sentences = generate_paraphrase(
    input_sentence, model, tokenizer, num_return_sequences=4
)

print(f"Original: {input_sentence}")
for i, paraphrase in enumerate(paraphrased_sentences, 1):
    print(f"Paraphrase {i}: {paraphrase}")

Original: Climate change is one of the most pressing issues of our time.
Paraphrase 1: Climate change is one of the most pressing issues of our time.
Paraphrase 2: Climate change is one of the most pressing issues of our time.
Paraphrase 3: Climate change is one of the most pressing issues of our time.
Paraphrase 4: Climate change is one of the most pressing issues of our time .
